In [ ]:
# Colab setup (uncomment if running in Google Colab)
# !pip install -q pandas numpy lightgbm xgboost optuna scikit-learn
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/sentinel-poc/project_sentinel

# Model Training: Cascaded Two-Stage Architecture

This notebook trains the cascaded Stage 1 and Stage 2 models for the Early Warning System.

## Architecture
- **Stage 1 (High-Recall Filter)**: Uses high-frequency, continuously available vital signs and readily available demographics to flag at-risk patients early. Designed to maximize recall and generate a dynamic risk score.
- **Stage 2 (High-Precision Classifier)**: For patients flagged by Stage 1, incorporates slower-turnaround laboratory values (e.g., Creatinine, BUN) and comprehensive data to confirm risk. Designed to maximize precision and reduce alarm fatigue.

In [ ]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('..'))
from src import models, features

PROCESSED_DATA_DIR = '../data/processed'
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Load the chronological splits produced by notebook 04 (features + labels together)
print("Loading train / val splits...")
train_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'train.parquet'))
val_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'val.parquet'))
print(f"train: {train_df.shape}  |  val: {val_df.shape}")

# Feature sets: Stage 1 = curated vitals/demographics; Stage 2 = all engineered features
stage1_feats = features.get_stage1_features()
stage2_feats = features.get_all_features(train_df)
print(f"Stage-1 features: {len(stage1_feats)}  |  Stage-2 features: {len(stage2_feats)}")

In [ ]:
print("Training Stage 1 models (vitals-only screener, one per horizon)...")
stage1_models = models.train_stage1_models(
    train_df, val_df, feature_cols=stage1_feats, horizons=[6, 12, 24]
)
print("Stage 1 trained for horizons:", list(stage1_models.keys()))

In [ ]:
print("Training Stage 2 models (LightGBM + XGBoost, Optuna-tuned)...")
# n_optuna_trials defaults to 50; lower it for a quick demo run if needed.
stage2_models = models.train_stage2_models(
    train_df, val_df, feature_cols=stage2_feats, horizons=[6, 12, 24], n_optuna_trials=50
)
print("Stage 2 trained for horizons:", list(stage2_models.keys()))

In [ ]:
print("Saving all models to disk...")
models.save_models(stage1_models, stage2_models, MODELS_DIR)
print("Models saved successfully.")

In [ ]:
# Conformal abstention threshold — the COMPOSER-style "I don't know" safety layer.
# Split-conformal (LAC): fit a threshold on the VALIDATION split using the SAME raw
# probabilities the clinical alerts use, for the Stage-2 LightGBM @ 24h model that
# notebook 07 explains. Scores inside the band [1-qhat, qhat] are abstained on.
from src.conformal import fit_conformal_threshold, indeterminate_rate

CONFORMAL_ALPHA = 0.10  # 90% coverage; lower alpha => wider band => abstains more often
conformal_qhat = None

if 24 in stage2_models:
    _m = stage2_models[24]["lgbm"]["model"]
    _p_val = _m.predict_proba(val_df[stage2_feats])[:, 1]
    _y_val = val_df["aki_label_24h"].values
    conformal_qhat = fit_conformal_threshold(_y_val, _p_val, alpha=CONFORMAL_ALPHA)
    lo, hi = 1 - conformal_qhat, conformal_qhat
    rate = indeterminate_rate(_p_val, conformal_qhat)
    print(f"Conformal qhat={conformal_qhat:.3f}  band=[{lo:.3f}, {hi:.3f}]  "
          f"val abstention rate={rate:.1%}")
    if lo > hi:
        print("Note: band empty -> the model is confident enough to never abstain at "
              "this alpha. Expected on the 100-patient demo (overconfident). Lower "
              "CONFORMAL_ALPHA to force a wider gray zone for the demo if you want one.")
else:
    print("No 24h Stage-2 model trained; skipping conformal fit.")

In [ ]:
# Save a self-describing metadata sidecar so the models can be loaded correctly
# on any machine (PC / Drive / HF). Records the trained feature lists, the exact
# library versions, the git commit of the training code, the alert threshold, and
# the conformal abstention threshold. Stdlib only — no new dependencies.
import json
import subprocess
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError


def _ver(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


def _git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        return None  # not a git checkout (e.g. fresh Colab download)


metadata = {
    "horizons": [6, 12, 24],
    "alert_threshold": 0.35,  # mirrors src.explain._DEFAULT_THRESHOLD
    "conformal_alpha": CONFORMAL_ALPHA,
    "conformal_qhat_stage2_24h": conformal_qhat,  # None if band/horizon unavailable
    "stage1_features": stage1_feats,
    "stage2_features": stage2_feats,
    "library_versions": {
        p: _ver(p)
        for p in ["lightgbm", "xgboost", "scikit-learn", "numpy", "pandas", "shap"]
    },
    "python": sys.version.split()[0],
    "git_commit": _git_commit(),
}

meta_path = Path(MODELS_DIR) / "model_metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2))
print(f"Wrote {meta_path}")
print(json.dumps(metadata, indent=2))

## Training Summary

We have successfully trained the cascaded models:
- **Stage 1 Models** are built on continuously available features (Vitals + Demographics).
- **Stage 2 Models** utilize the full feature set including laboratory results, tuned using Optuna.
- All models are persisted in the `models/` directory for evaluation and subsequent deployment.